### Verify vis.npz
Sanity-checks a visibility `.vis.npz` exported by `ExportMS.py` before trusting it for frank fitting / injection-recovery: structure, corruption (NaN/Inf, truncation), value ranges, and a weight-vs-RMS cross-check against `diskdictionary_DSHARP.py`.

In [ ]:
import sys
import numpy as np

sys.path.append(r"D:/CPD_MPIA/DSHARP_rep")
import diskdictionary_DSHARP as disk

target = 'HD143006'
path = r"D:/CPD_MPIA/DSHARP_rep/data/HD143006_continuum_spavg.vis.npz"

dat = np.load(path)
print(f"Loaded {path}")
print("keys:", dat.files)


In [ ]:
n_preview = 5
print(f"first {n_preview} rows:")
print(f"{'u':>14} {'v':>14} {'Vis (real)':>14} {'Vis (imag)':>14} {'Wgt':>14}")
for i in range(min(n_preview, len(dat['u']))):
    print(f"{dat['u'][i]:14.4f} {dat['v'][i]:14.4f} "
          f"{dat['Vis'][i].real:14.6f} {dat['Vis'][i].imag:14.6f} "
          f"{dat['Wgt'][i]:14.4f}")


### 1. Structural checks

In [ ]:
expected_keys = {'u', 'v', 'Vis', 'Wgt'}
missing = expected_keys - set(dat.files)
assert not missing, f"missing keys {missing} (found {dat.files})"
print(f"OK: found all expected keys {sorted(dat.files)}")

u, v, Vis, Wgt = dat['u'], dat['v'], dat['Vis'], dat['Wgt']
n = len(u)
print(f"n_vis = {n}")

for name, arr in [('u', u), ('v', v), ('Vis', Vis), ('Wgt', Wgt)]:
    assert len(arr) == n, f"{name} has length {len(arr)}, expected {n}"
print("OK: all arrays same length")

assert n > 0, "zero visibility points -- file is empty/corrupted"


### 2. Finite-value checks (catches truncated writes / garbage floats)

In [ ]:
ok = True
for name, arr in [('u', u), ('v', v), ('Vis', Vis), ('Wgt', Wgt)]:
    n_bad = (~np.isfinite(arr)).sum()
    status = "OK" if n_bad == 0 else "FAIL"
    if n_bad:
        ok = False
    print(f"{status}: {name} -- {n_bad}/{n} non-finite (NaN/Inf) values")


### 3. Value sanity (order-of-magnitude checks for ALMA continuum data)

In [ ]:
uvdist = np.sqrt(u**2 + v**2)
print(f"uvdist range: {uvdist.min():.1f} - {uvdist.max():.1f} lambda")
if uvdist.max() < 1e3 or uvdist.max() > 1e8:
    print("WARN: uv-distance range looks unusual for ALMA continuum "
          "(expected roughly 1e3-1e7 lambda) -- check units")

vis_amp = np.abs(Vis)
print(f"|Vis| range: {vis_amp.min():.3e} - {vis_amp.max():.3e} Jy "
      f"(median {np.median(vis_amp):.3e})")
if vis_amp.max() > 10:
    print("WARN: max |Vis| > 10 Jy -- unusually bright for a protoplanetary "
          "disk continuum, check flux units/flagging")

if np.any(Wgt <= 0):
    n_bad = (Wgt <= 0).sum()
    print(f"FAIL: {n_bad}/{n} non-positive weights (should all be > 0)")
    ok = False
else:
    print(f"OK: Wgt range: {Wgt.min():.3e} - {Wgt.max():.3e}, all positive")


### 4. Weight-vs-RMS cross-check (catches a weight-scale/units mismatch)

In [ ]:
implied_rms_Jy = 1 / np.sqrt(np.sum(Wgt))
implied_rms_uJy = implied_rms_Jy * 1e6
known_rms_uJy = disk.disk[target]['RMS']

ratio = implied_rms_uJy / known_rms_uJy
print(f"weight-implied thermal noise: {implied_rms_uJy:.2f} uJy")
print(f"known image RMS (diskdictionary_DSHARP.py): {known_rms_uJy:.2f} uJy")
print(f"ratio: {ratio:.2f}x")

if 0.3 <= ratio <= 3:
    print("OK: same order of magnitude -- weights look consistent with the measured image RMS")
else:
    print("WARN: weight-implied noise is off from the known image RMS by "
          f"{ratio:.1f}x -- weights may be on the wrong scale (e.g. missing "
          "a statwt-style rescaling upstream). Note this is only a rough "
          "check: weight-implied noise is a per-visibility theoretical "
          "estimate, not directly the same quantity as image RMS, so some "
          "difference is expected -- but an order-of-magnitude gap is a red flag.")


### 5. Verdict

In [ ]:
print("PASS" if ok else "FAIL -- see checks above")
